# Simple Medallion Architecture Demo with SQL

This notebook shows a simple medallion architecture demo with SQL using the sample NYC Taxi dataset, available by default on Databricks.

Prerequisites:
- Create the `databricks_training_medallion` catalog with `bronze`, `silver` and `gold` schemas.

Notebook summary:
- ingest raw taxi trip data into the tables of the `bronze` layer (schema)
- apply data transformations: save data with flagged rides and weekly statistics into `silver` layer tables
- integrate the data from the silver layer to prepare `gold` layer table with a comprehensive view of the top three highest-fare rides

### Bronze layer: Raw data ingestion
Ingest raw taxi trip data, with a basic data quality check applied to ensure trip distances are positive.

In [0]:
CREATE OR REPLACE TABLE databricks_training_medallion.bronze.taxi_raw_records AS
SELECT *
FROM samples.nyctaxi.trips
WHERE trip_distance > 0.0;

In [0]:
SELECT * FROM databricks_training_medallion.bronze.taxi_raw_records;

### Silver layer

The Silver layer creates two tables: 
1. **Flagged rides**: This table identifies potentially suspicious rides based on fare and distance criteria.

In [0]:
CREATE OR REPLACE TABLE databricks_training_medallion.silver.flagged_rides AS
SELECT
  date_trunc("week", tpep_pickup_datetime) AS week,
  pickup_zip AS zip,
  fare_amount,
  trip_distance
FROM
  databricks_training_medallion.bronze.taxi_raw_records
WHERE ((pickup_zip = dropoff_zip AND fare_amount > 50) OR
       (trip_distance < 5 AND fare_amount > 50));

In [0]:

SELECT * FROM databricks_training_medallion.silver.flagged_rides;

2. **Weekly statistics**: This silver table calculates weekly average fares and trip distances. 

In [0]:
CREATE OR REPLACE TABLE databricks_training_medallion.silver.weekly_stats AS
SELECT
  date_trunc("week", tpep_pickup_datetime) AS week,
  AVG(fare_amount) AS avg_amount,
  AVG(trip_distance) AS avg_distance
FROM
  databricks_training_medallion.bronze.taxi_raw_records
GROUP BY week
ORDER BY week ASC;

In [0]:
SELECT * FROM databricks_training_medallion.silver.weekly_stats;

### Gold layer: Top N rides

Here, these silver tables are integrated to provide a comprehensive view of the top three highest-fare rides.

In [0]:
CREATE OR REPLACE TABLE databricks_training_medallion.gold.top_n_rides AS
SELECT
  ws.week,
  ROUND(ws.avg_amount, 2) AS avg_amount,
  ROUND(ws.avg_distance, 3) AS avg_distance,
  fr.fare_amount,
  fr.trip_distance,
  fr.zip
FROM
  databricks_training_medallion.silver.flagged_rides fr
LEFT JOIN databricks_training_medallion.silver.weekly_stats ws ON ws.week = fr.week
ORDER BY fr.fare_amount DESC
LIMIT 3;

In [0]:
SELECT * FROM databricks_training_medallion.gold.top_n_rides;